# ADALL Practical Test Revision: Regression Workflow

## Single-session version

This notebook is designed for **one revision session**.

The main route is short. Read the optional sections only after you finish the main route.

### Session target
By the end of this session, you should be able to:

1. explain why this is a **regression** task,
2. create the target `num_failed_subjects`,
3. avoid leakage from `G1`, `G2`, and `G3`,
4. create a dataset profile for LLM review,
5. train models using a pipeline,
6. compare results using CV and holdout MAE,
7. write a short judgement-based answer.

### Suggested timing

| Part | Time |
|---|---:|
| Admin and test reminder | 10 to 15 min |
| Main route in this notebook | 75 to 90 min |
| Recap and Q&A | 15 to 20 min |
| Optional practice | After class |

<div style="background-color:#eaf4ff; border-left:6px solid #2b7bbb; padding:12px;">
<strong>Main idea:</strong> The practical test is not mainly about writing many new lines of code. It is about knowing what the code is doing, running the right block, checking the result, and explaining your decision clearly.
</div>


## What the previous lecturer focused on

The transcript suggests that this revision should focus on the following points:

| Focus area | What students should pay attention to |
|---|---|
| Regression only | The practical test focuses on predicting a number, not classification. |
| Read instructions carefully | If a cell says “run only”, do not edit it unnecessarily. |
| Dataset profile and payload text | You should know how to summarise the dataset for LLM-assisted review. |
| Leakage | `G1`, `G2`, and `G3` directly define the target, so they must not be used as predictors. |
| Train/test split | Keep the test set for final checking. Use a fixed `random_state`. |
| Pipeline | Use the same preprocessing and modelling structure fairly. |
| Model comparison | Choose based on evidence, not because a model sounds more advanced. |

<div style="background-color:#fff8e6; border-left:6px solid #d69b00; padding:12px;">
<strong>Revision stance:</strong> You are practising judgement. Do not accept every LLM recommendation automatically. Check whether the recommendation makes sense for the task.
</div>


## Step 0. Setup

Run this cell first.

Do not edit the imports unless you get a clear library error.


In [ ]:
# Core libraries
import os
import numpy as np
import pandas as pd

# Visualisation
import matplotlib.pyplot as plt

# Modelling
from sklearn.model_selection import train_test_split, ShuffleSplit, cross_validate
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor

from xgboost import XGBRegressor


### What to check after Step 0

You should see no error.

If there is an error, it is usually because a library is not installed in the runtime. In Colab, install the missing library, restart the runtime if needed, then run the setup again.


## Step 1. Load the dataset and do quick checks

The practical test may provide the loading code for you.

When the instruction says **run the cell**, just run it first. Do not rewrite the loading code unless the question asks you to.

In this revision notebook, we use the student performance dataset.


In [ ]:
# Load data from KaggleHub
# In the actual practical test, the dataset path/code may be provided to you.

import kagglehub

path = kagglehub.dataset_download("devansodariya/student-performance-data")
print("Downloaded to:", path)
print("Files:", os.listdir(path))

df = pd.read_csv(os.path.join(path, "student_data.csv"))
df.head()


In [ ]:
# Quick checks
print("Shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())
print("\nData info:")
df.info()


### What to check after Step 1

Check these before moving on:

1. Does the dataset load correctly?
2. How many rows and columns are there?
3. Are `G1`, `G2`, and `G3` present?
4. Are there missing values?
5. Which columns are numeric and which are categorical?

Short answer format:

```text
The dataset has ___ rows and ___ columns. It contains a mix of numeric and categorical columns. The grade columns are G1, G2 and G3.
```


## Step 2. Create a dataset profile for LLM review

A dataset profile is a text summary of the dataset.

You use it so the LLM does not need the full dataset. It can review the structure, missing values, data types, correlations, and possible risks.

### What the payload should include

- dataset shape,
- column data types,
- numeric summary,
- categorical summary,
- missing values,
- unique value counts,
- simple correlation information,
- possible leakage clues.

<div style="background-color:#fff0f0; border-left:6px solid #d93636; padding:12px;">
<strong>Important:</strong> The LLM recommendation is not automatically correct. You still need to decide whether each recommendation is sensible.
</div>


In [ ]:
from io import StringIO

buffer = StringIO()

buffer.write("=== SHAPE ===\n")
buffer.write(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}\n\n")

buffer.write("=== DTYPES ===\n")
buffer.write(df.dtypes.to_string())
buffer.write("\n\n")

buffer.write("=== NUMERIC DESCRIBE ===\n")
buffer.write(df.describe().to_string())
buffer.write("\n\n")

buffer.write("=== CATEGORICAL DESCRIBE ===\n")
cat_cols = df.select_dtypes(include="object").columns.tolist()
if cat_cols:
    buffer.write(df[cat_cols].describe().to_string())
else:
    buffer.write("No categorical columns")
buffer.write("\n\n")

buffer.write("=== NULL SUMMARY ===\n")
null_summary = df.isna().sum().to_frame("null_count")
null_summary["null_pct"] = null_summary["null_count"] / len(df)
buffer.write(null_summary.to_string())
buffer.write("\n\n")

buffer.write("=== UNIQUE VALUES PER COLUMN ===\n")
buffer.write(df.nunique().to_frame("unique_count").to_string())
buffer.write("\n\n")

buffer.write("=== CORRELATIONS: NUMERIC ONLY ===\n")
buffer.write(df.corr(numeric_only=True).round(3).to_string())
buffer.write("\n\n")

buffer.write("=== POSSIBLE UNIQUE-ID COLUMNS ===\n")
unique_id_cols = df.columns[df.nunique() == len(df)].tolist()
buffer.write(str(unique_id_cols))
buffer.write("\n")

payload_text = buffer.getvalue()
print(payload_text)


### What to check after Step 2

Your payload should be readable.

Check that it contains enough information for a reviewer to answer:

1. What is the dataset size?
2. What are the main column types?
3. Are there missing values?
4. Are there possible leakage columns?
5. Are `G1`, `G2`, and `G3` strongly related to the target you plan to create?


## Step 3. Ask the LLM a small, focused question

Do not ask the LLM to solve everything at once.

Start with a small review question:

> “Based on this dataset profile and the target definition, what data risks should I check before modelling?”

The aim is to get a review, not to blindly follow instructions.


In [ ]:
RUN_API_CELLS = True
OPENAI_MODEL = 'gpt-5.4-nano'
client = None

# Only use this section if your tutor has asked you to call the API from Colab.
# You must first save OPENAI_API_KEY in Colab Secrets.

if RUN_API_CELLS == True:
    from google.colab import userdata
    from openai import OpenAI

    api_key = userdata.get('OPENAI_API_KEY')
    client = OpenAI(api_key=api_key)
    print('OpenAI client is ready.')
else:
    print('Manual chatbot mode. Copy the prompts when they appear.')

In [ ]:
llm_prompt = f"""
You are helping me review a regression dataset for a practical test revision.

Task:
I want to predict the number of failed subjects.
A subject is failed when its grade is below 10.
The grade columns are G1, G2 and G3.

Dataset profile:
{payload_text}

Please answer in three sections:
1. Data quality issues to check before modelling.
2. Columns that may cause leakage, with reasons.
3. Which recommendations need human judgement before applying.

Keep the answer concise. Do not write modelling code yet.
"""

print(llm_prompt)


### Optional: use the OpenAI API

Run this only if your Colab secret `OPENAI_API_KEY` is already set.

Otherwise, copy the printed prompt above and paste it into the approved LLM tool.


In [ ]:
if client is not None:
    response = client.responses.create(
        model=OPENAI_MODEL,
        input=llm_prompt
    )
    print('\nLLM response:')
    print(response.output_text)
else:
    print('\nNo API response because client is not connected. Copy the prompt above into your chatbot.')


### Decision point after LLM review

Write a short judgement note:

```text
I agree with ___ because ___.
I disagree with ___ because ___.
I will not use G1, G2, and G3 as predictors because they directly define the target.
```

This is the key learning point. The LLM can suggest, but you must judge.


## Step 4. Create the target: number of failed subjects

This is the main target engineering step.

A subject is counted as failed when the grade is **less than 10**.

```text
G1 < 10 means failed G1
G2 < 10 means failed G2
G3 < 10 means failed G3
```

The target is the count of failed subjects across `G1`, `G2`, and `G3`.

Although the target values are usually 0, 1, 2, and 3, this is still treated as **regression** here because you are predicting a number.

### Common mistakes

| Mistake | Why it matters |
|---|---|
| Using `<= 10` instead of `< 10` | It changes the target definition. |
| Forgetting to check the target range | The target should normally be 0 to 3. |
| Using `G1`, `G2`, `G3` later as predictors | This causes leakage. |


In [ ]:
grade_cols = ["G1", "G2", "G3"]

# Count how many of G1, G2, G3 are below 10 for each student
df["num_failed_subjects"] = (df[grade_cols] < 10).sum(axis=1)

# Sanity check
print(df[grade_cols + ["num_failed_subjects"]].head())
print("\nTarget value counts:")
print(df["num_failed_subjects"].value_counts().sort_index())
print("\nTarget proportions:")
print(df["num_failed_subjects"].value_counts(normalize=True).sort_index())


In [ ]:
# Visual check of target distribution
plt.figure(figsize=(7, 4))
plt.hist(df["num_failed_subjects"], bins=[-0.5, 0.5, 1.5, 2.5, 3.5], edgecolor="black")
plt.xticks([0, 1, 2, 3])
plt.title("Target distribution: number of failed subjects")
plt.xlabel("num_failed_subjects")
plt.ylabel("count")
plt.show()


### What to check after Step 4

Check:

1. Are the target values only 0, 1, 2, and 3?
2. Is the target imbalanced?
3. Are there very few cases for any target value?

Short answer format:

```text
The target ranges from ___ to ___. Most students have __ failed subjects. This matters because rare target values may be harder to predict.
```


## Step 5. Define X and y, then remove leakage columns

This is a very important test point.

`G1`, `G2`, and `G3` are used to calculate `num_failed_subjects`.

If you include them as predictors, the model is indirectly given the answer.

That is leakage.

### Simple example

If I already know all three subject marks, I can calculate the number of failed subjects directly.

So the model does not need to learn from behaviour, background, or other features. It is just using the answer source.

<div style="background-color:#fff0f0; border-left:6px solid #d93636; padding:12px;">
<strong>Rule for this task:</strong> Drop <code>G1</code>, <code>G2</code>, and <code>G3</code> from X after creating the target.
</div>


In [ ]:
target_col = "num_failed_subjects"
leakage_cols = ["G1", "G2", "G3"]

X = df.drop(columns=[target_col] + leakage_cols)
y = df[target_col]

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Columns removed for leakage:", leakage_cols)


### Decision point after Step 5

Write this in your answer if asked about leakage:

```text
I removed G1, G2 and G3 because the target is calculated from these columns. Keeping them would leak the answer into the model and make the evaluation unrealistic.
```


## Step 6. Train/test split

The test set is used only for final checking.

For normal regression, we usually do not stratify.

However, this target is a small integer count: 0, 1, 2, 3. It behaves partly like groups. In this revision example, we use `stratify=y` so the train and test sets keep similar target proportions.

### Caveat
If the target has rare values, stratified splitting may fail or create very small groups. Always check the value counts.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.10,
    random_state=42,
    stratify=y
)

print("Train shape:", X_train.shape, y_train.shape)
print("Test shape:", X_test.shape, y_test.shape)

print("\nTrain target proportions:")
print(y_train.value_counts(normalize=True).sort_index())

print("\nTest target proportions:")
print(y_test.value_counts(normalize=True).sort_index())


### What to check after Step 6

Check that the train and test target proportions are reasonably similar.

Short answer format:

```text
The train and test sets have similar target proportions, so the holdout test set is a reasonable final check.
```


## Step 7. Build the preprocessing pipeline

The dataset has both numeric and categorical columns.

A pipeline helps you apply the same preparation steps during training and prediction.

### What happens here

| Column type | What we do |
|---|---|
| Numeric columns | Pass through as they are |
| Categorical columns | One-hot encode them |

### Why this matters

A pipeline reduces mistakes because the preprocessing is fitted on the training data and then applied consistently to validation and test data.

This avoids common errors such as preparing train and test data differently.


In [ ]:
num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ('num', 'passthrough', num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
    ]
)

### What to check after Step 7

Check:

1. Did the code correctly separate numeric and categorical columns?
2. Are categorical columns being encoded?
3. Is the preprocessing inside the pipeline later?

Short answer format:

```text
The pipeline uses a ColumnTransformer so numeric and categorical columns are handled consistently before modelling.
```


## Step 8. Train and compare models using CV

We compare two tree-based models:

- Random Forest
- XGBoost

The main point is not to memorise the model code.

The main point is to compare fairly using the same:

1. train/test split,
2. preprocessing,
3. CV method,
4. metric.

### CV reminder

Cross-validation repeats model checking on different splits of the training data.

Here we use `ShuffleSplit`:

```text
Round 1: use part of training data for validation, train on the rest
Round 2: reshuffle, use another validation part, train on the rest
...
Average the validation scores
```

We also record train MAE so we can check overfitting.


In [ ]:
# Keep this small enough for a single revision session.
# The goal is model comparison, not exhaustive tuning.

cv = ShuffleSplit(n_splits=5, test_size=0.20, random_state=42)

models = {
    "Random Forest": RandomForestRegressor(
        n_estimators=200,
        max_depth=8,
        random_state=42,
        n_jobs=-1
    ),
    "XGBoost": XGBRegressor(
        n_estimators=200,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1
    )
}

results = []
fitted_models = {}

for name, model in models.items():
    pipe = Pipeline([
        ("preprocessor", preprocessor),
        ("regressor", model)
    ])

    cv_out = cross_validate(
        pipe,
        X_train,
        y_train,
        cv=cv,
        scoring="neg_mean_absolute_error",
        return_train_score=True,
        n_jobs=-1
    )

    train_mae = -cv_out["train_score"]
    val_mae = -cv_out["test_score"]

    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    holdout_mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    results.append({
        "model": name,
        "cv_train_mae_mean": train_mae.mean(),
        "cv_train_mae_std": train_mae.std(),
        "cv_val_mae_mean": val_mae.mean(),
        "cv_val_mae_std": val_mae.std(),
        "train_val_gap": val_mae.mean() - train_mae.mean(),
        "holdout_mae": holdout_mae,
        "holdout_rmse": rmse,
        "holdout_r2": r2
    })

    fitted_models[name] = pipe

results_df = pd.DataFrame(results).sort_values("cv_val_mae_mean")
results_df


### What to check after Step 8

Check the result table in this order:

1. Which model has lower **CV validation MAE**?
2. Is the train MAE much lower than the validation MAE?
3. Does the holdout MAE support the CV result?
4. Is the improvement meaningful, or only tiny?

Short answer format:

```text
I would choose ___ because it has lower validation MAE, the train-validation gap is acceptable, and the holdout MAE supports the CV result.
```

Do not choose a model only because it sounds more advanced.


## Step 9. Plot actual vs predicted values

This plot helps you see whether the model is roughly following the true target values.

Because the target is 0 to 3, predictions may be non-integers.

That is normal for regression.

Only round predictions if the question asks for integer outputs.


In [ ]:
# Choose the model with the best CV validation MAE
best_model_name = results_df.iloc[0]["model"]
best_model = fitted_models[best_model_name]

print("Best model based on CV validation MAE:", best_model_name)

y_pred_test = best_model.predict(X_test)

# Sort by true target so the pattern is easier to see
order = np.argsort(y_test.to_numpy())
y_true_sorted = y_test.to_numpy()[order]
y_pred_sorted = y_pred_test[order]

plt.figure(figsize=(10, 5))
plt.plot(y_true_sorted, label="Actual", marker="o")
plt.plot(y_pred_sorted, label="Predicted", marker="x")
plt.title(f"Actual vs Predicted: {best_model_name}")
plt.xlabel("Test sample index sorted by actual target")
plt.ylabel("Number of failed subjects")
plt.legend()
plt.tight_layout()
plt.show()


### What to check after Step 9

Look for:

1. Does the predicted line roughly follow the actual line?
2. Does the model over-predict students with 0 failed subjects?
3. Does the model under-predict students with 2 or 3 failed subjects?
4. Are the largest errors concentrated in rare target values?

Short answer format:

```text
The model generally follows the actual values, but it struggles more with ___ because ___.
```


## Step 10. Final judgement answer

Use this structure for a short written response.

```text
I treated this as a regression task because the target is a numeric count: number of failed subjects.

I created the target by counting how many of G1, G2, and G3 are below 10.

I removed G1, G2, and G3 from the predictors because they directly define the target and would cause leakage.

I used a pipeline so that preprocessing and modelling are applied consistently.

Based on the CV validation MAE, train-validation gap, and holdout MAE, I would choose ___ because ___.
```

<div style="background-color:#eaf4ff; border-left:6px solid #2b7bbb; padding:12px;">
<strong>Recap:</strong> Good practical-test answers are short, evidence-based, and clear. You do not need to explain every line of code.
</div>


# Optional Reading

Read this only after the main route.

## Why this target is a slight edge case

The target `num_failed_subjects` has values like 0, 1, 2, and 3.

That can feel like classification because the values look like categories.

But in this notebook, we treat it as regression because the target is a count.

### Standard interpretation

| Situation | Usually use |
|---|---|
| Predict a continuous number, such as price | Regression |
| Predict a count, such as number of failed subjects | Often regression, sometimes count models |
| Predict a category, such as pass/fail | Classification |

### Edge case

If the task changes to “predict whether a student failed at least one subject”, then the target becomes:

```text
0 = no failed subject
1 = failed at least one subject
```

That becomes classification.

So the method depends on how the target is defined.


# Optional Reading: CV and train/test split

## Train/test split

A train/test split holds back some data for final checking.

The model should not use the test set during training or model selection.

## Cross-validation

CV checks the model several times using different validation splits inside the training data.

For ShuffleSplit, each round reshuffles the training data and creates a new validation split.

```text
Round 1: train on most of the training data, validate on a held-out part
Round 2: reshuffle, train again, validate on another held-out part
Round 3: reshuffle, train again, validate on another held-out part
```

This reduces the chance that your result is due to one lucky split.

## Stratification caveat

For normal regression, stratification is not usually used.

Here, the target is a small integer count. Using `stratify=y` helps keep the 0, 1, 2, 3 proportions similar between train and test.

If a target value is very rare, stratification may fail or give unstable results.


# Additional Practice

Do these after class or during revision.

## Practice 1: Explain leakage

Write a 3-line answer explaining why `G1`, `G2`, and `G3` should be removed.

## Practice 2: Improve one modelling choice

Try only one improvement:

- change Random Forest `max_depth`, or
- change XGBoost `max_depth`, or
- change XGBoost `learning_rate`.

Then compare the new result with the original result.

Do not tune many things at once.

## Practice 3: Ask the LLM to review your answer

Use this prompt:

```text
Here is my practical-test answer. Check if it is clear, evidence-based, and free from leakage mistakes. Do not rewrite everything. Point out only the top three improvements.
```

Paste your own answer below that prompt.
